In [6]:
import numpy as np
import os
from qforge.utils.terminal_plot import TerminalPlotter
from qforge import QubitEngine
from qforge.core.gate_engine import GateEngine

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [9]:
def simulate_and_plot(g_eng, gate_type, duration, drive_amp, w01):
    """Simulates the gate dynamics and prints the population table and graph."""
    print(f"\n[EVOLUTION]: Simulating {gate_type}-Gate dynamics...")
    
    drives = [{"target": 0, "type": gate_type, "amplitude": drive_amp, "frequency": w01, "phase": 0.0}]
    
    res = g_eng.simulate_n_qubit_dynamics(
        ["Q1"], f"Calibrated_{gate_type}", duration, [], drives, "0", steps=50
    )
    
    times, p0, p1 = res["times"], res["populations"]["0"], res["populations"]["1"]
    p_leak = 1.0 - np.array(p0) - np.array(p1)
    
    print("\n   Time (ns) | P(|0>) | P(|1>) | P(|2>) (Leakage)")
    print("   ----------------------------------------------")
    for i in range(0, len(times), 10):
        print(f"   {times[i]:9.2f} |  {p0[i]:.3f} |  {p1[i]:.3f} |  {p_leak[i]:.6f}")
    print(f"   {times[-1]:9.2f} |  {p0[-1]:.3f} |  {p1[-1]:.3f} |  {p_leak[-1]:.6f}\n")
    
    TerminalPlotter.plot_time_evolution(
        times=times,
        expectations=[p0, p1, p_leak],
        labels=['P(|0>)', 'P(|1>)', 'P(|2>) (Leakage)'],
        title=f"Automated Single Qubit {gate_type}-Gate | Duration: {duration:.2f} ns"
    )

def main():
    print("============================================================")
    print(" Example 14: Single Qubit Gate (X & H) Automated Calibration")
    print("============================================================")
    
    q_eng = QubitEngine()
    g_eng = GateEngine()

    q_eng.create_qubit("transmon", "Q1", {"EJ": 15.0, "EC": 0.3, "truncated_dim": 3})
    
    qubit = q_eng.get_qubit("Q1")
    evals = qubit.eigensys(evals_count=2)[0]
    w01 = evals[1] - evals[0]
    
    drive_amp = 0.02 
    print(f"\n[CALIBRATION]: Finding optimal durations for Amp = {drive_amp} GHz...")

    best_x_dur, x_metric = g_eng.calibrate_gate(
        q1_name="Q1", gate_type="X", parameter="duration", amplitude=drive_amp
    )
    print(f"  -> Calibrated X-Pulse Duration: {best_x_dur:.2f} ns (Fidelity: {x_metric:.4f})")

    best_h_dur, h_metric = g_eng.calibrate_gate(
        q1_name="Q1", gate_type="H", parameter="duration", amplitude=drive_amp
    )
    print(f"  -> Calibrated H-Pulse Duration: {best_h_dur:.2f} ns (Fidelity: {h_metric:.4f})")

    # Execute simulation and plotting for both gates
    simulate_and_plot(g_eng, "X", best_x_dur, drive_amp, w01)
    simulate_and_plot(g_eng, "H", best_h_dur, drive_amp, w01)

if __name__ == "__main__":
    main()

 Example 14: Single Qubit Gate (X & H) Automated Calibration

[CALIBRATION]: Finding optimal durations for Amp = 0.02 GHz...
  [CACHE HIT] Returning saved duration for X on Q1
  -> Calibrated X-Pulse Duration: 55.20 ns (Fidelity: 0.9975)
  [CACHE HIT] Returning saved duration for H on Q1
  -> Calibrated H-Pulse Duration: 27.45 ns (Fidelity: 0.9982)

[EVOLUTION]: Simulating X-Gate dynamics...

   Time (ns) | P(|0>) | P(|1>) | P(|2>) (Leakage)
   ----------------------------------------------
        0.00 |  1.000 |  0.000 |  0.000000
       11.27 |  0.997 |  0.003 |  0.000000
       22.53 |  0.805 |  0.194 |  0.000255
       33.80 |  0.146 |  0.853 |  0.000971
       45.06 |  0.004 |  0.996 |  0.000048
       55.20 |  0.003 |  0.997 |  0.000000

                 Automated Single Qubit X-Gate | Duration: 55.20 ns             
    ┌┬─────────────────┬──────────────────┬─────────────────┬─────────────────┬┐
1.00┼ •• P(|0>)           •───────────────┼──────────────••••••••••••••••••••••┤
  

In [11]:
"""
Gate Simulation Example
=======================

This example demonstrates how to use the qforge API to:
1. Create a Transmon qubit.
2. Analyze its energy spectrum.
3. Simulate a quantum gate (X-gate / Pi-pulse).
4. Visualize the dynamics directly in the terminal.
"""

def run_simulation():
    print("\n=== qforge Gate Simulation Example ===\n")

    # 1. Initialize Engines
    qubit_engine = QubitEngine()

    # 2. Create a Transmon Qubit
    print("-> Creating Transmon...")
    qubit = qubit_engine.create_qubit(
        qubit_type="transmon",
        name="example_qubit",
        params={"EJ": 15.0, "EC": 0.3}
    )

    # 3. Compute and Plot Spectrum
    print("-> Computing Spectrum...")
    spectrum = qubit_engine.compute_spectrum(qubit, n_levels=5, subtract_ground=True)
    TerminalPlotter.plot_spectrum(spectrum, title="Energy Spectrum (Relative)")

    # 4. Simulate X Gate (Pi Pulse)
    print("\n-> Simulating X Gate (Pi-pulse)...")
    gate_engine = GateEngine()
   
    duration = 40.0 # ns
    
    result = gate_engine.simulate_dynamics(
        qubit_name="example_qubit",
        gate_type="X",
        duration=duration,
        noise_model="realistic",
        steps=100
    )

    # 5. Visualize Dynamics
    print("-> Plotting Dynamics...")
    TerminalPlotter.plot_time_evolution(
        times=result["times"],
        expectations=result["expectations"],
        labels=result["labels"],
        title="Rabi Oscillation (X Gate)"
    )
    
    print("\n=== Simulation Complete ===")

if __name__ == "__main__":
    run_simulation()



=== qforge Gate Simulation Example ===

-> Creating Transmon...
-> Computing Spectrum...
                             Energy Spectrum (Relative)                         
    ┌┬─────────────────┬──────────────────┬─────────────────┬─────────────────┬┐
20.4┼ ▞▞ Energy Levels ┼──────────────────┼─────────────────┼─────────────────•┤
    ││                 │                  │                 │           ••••••││
    ││                 │                  │                 │     ••••••      ││
17.0┼┼─────────────────┼──────────────────┼─────────────────••••••────────────┼┤
    ││                 │                  │             ••••│                 ││
13.6┼┼─────────────────┼──────────────────┼────────•••••────┼─────────────────┼┤
    ││                 │                  │    ••••         │                 ││
    ││                 │                  •••••             │                 ││
10.2┼┼─────────────────┼───────────────•••┼─────────────────┼─────────────────┼┤
    ││             